# MBMP coherence — visualisation

Reads the output of `compute_coherence.py` (`<ROI>/<stack>/coherence_w<L>m.npz`) and, **for every stack separately**, shows

1. the mean of the whole **MBMP correlation matrix** (3 pols x Nb baselines = 21 x 21): mean |γ| and phase of the mean γ
   - block order as in BPS: `index = pol * Nb + acquisition`, pols `HH`, `XP = (HV+VH)/2`, `VV`
2. **acquisition filter**, per polarisation: an acquisition is dropped when the mean of its 3-day |γ| (its pairs with the
   acquisitions 3 days before / after, spatial mean over the valid cells) is below mean - `N_STD` x std of these
   values over the acquisitions of the stack (threshold computed per stack and per polarisation)
3. for each single polarisation **HH, HV, VH, VV** a grid of the kept acquisitions: |γ| maps above the diagonal,
   intensity [dB] on the diagonal
4. **RGB of the temporal decorrelation** for HH, XP, VV: for every kept anchor acquisition *i* whose +3, +6 and +9 day
   partners are also kept, R = |γ(i, i+3d)|, G = |γ(i, i+6d)|, B = |γ(i, i+9d)| (linear, 0-1)

Data are flattened (LUT `flatteningPhaseScreen`), no SKP calibration. Maps are in radar geometry, axes in pixels of the L1C crop (azimuth = rows, slant range = columns; cell centres from
the npz `az_index` / `rg_index`) at the BPS subsampled grid (half window); cells with less than `MIN_VALID` of the window inside the common area are masked.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

ROI_DIR = Path('Lac_Tele')   # folder written by extract_l1c_roi.py
WINDOW_M = 250               # averaging window used in compute_coherence.py [m]
MIN_VALID = 0.5              # minimum share of valid pixels in the averaging window
N_STD = 1.0                  # acquisitions with mean 3-day |γ| below mean - N_STD * std are dropped (per pol)
RGB_LAGS_DAYS = (3, 6, 9)    # temporal baselines shown as R, G, B
RGB_POLS = ['HH', 'XP', 'VV']

files = sorted(ROI_DIR.glob(f'*/coherence_w{WINDOW_M:g}m.npz'))
print(f'{len(files)} stacks with a {WINDOW_M:g} m coherence:')
for f in files:
    print('  ', f.parent.name)

In [ ]:
def load_stack(path):
    meta = json.loads(path.with_suffix('.json').read_text())
    with np.load(path) as d:
        st = {k: d[k] for k in d.files}
    st['meta'] = meta
    st['name'] = path.parent.name
    st['good'] = st['valid_fraction'] >= MIN_VALID
    win = meta['window']
    # maps in radar geometry with crop pixel axes: one output cell = d_az x d_rg crop pixels
    d_az, d_rg = np.diff(st['az_index'][:2])[0], np.diff(st['rg_index'][:2])[0]
    st['extent'] = [st['rg_index'][0] - d_rg / 2, st['rg_index'][-1] + d_rg / 2,
                    st['az_index'][-1] + d_az / 2, st['az_index'][0] - d_az / 2]
    # imshow aspect = ground height / width of one crop pixel (azimuth pixel vs ground range pixel)
    st['px_aspect'] = (win['output_spacing_az_m'] / d_az) / (win['output_spacing_rg_m'] / d_rg)
    st['aspect'] = win['output_spacing_az_m'] / win['output_spacing_rg_m']   # ground height / width of one cell
    st['days'] = np.array([np.datetime64(d) for d in meta['dates']])
    nb = len(meta['dates'])
    # (Naz, Nrg, Nb, Nb) complex coherence per pol; XP = diagonal block of the MBMP matrix
    st['coh'] = {pol: st['coh_pol'][:, :, p] for p, pol in enumerate(meta['coh_pols'])}
    xp = meta['mbmp_pols'].index('XP')
    st['coh']['XP'] = st['mbmp_corr'][:, :, xp * nb:(xp + 1) * nb, xp * nb:(xp + 1) * nb]
    return st


def lag_days(st, i, j):
    return int((st['days'][j] - st['days'][i]) / np.timedelta64(1, 'D'))


def show_map(ax, st, img, labels=True, **kw):
    """Draw a (Naz, Nrg) image, or (Naz, Nrg, 3|4) RGB(A), in radar geometry with crop pixel axes (invalid cells masked)."""
    if img.ndim == 2:
        img = np.where(st['good'], img, np.nan)
    im = ax.imshow(img, extent=st['extent'], aspect=st['px_aspect'], interpolation='nearest', **kw)
    ax.tick_params(labelsize=6)
    if labels:
        ax.set_xlabel('range pixel', fontsize=7)
        ax.set_ylabel('azimuth pixel', fontsize=7)
    return im


def three_day_coherence(st, pol):
    """Per acquisition: mean of the spatially averaged |γ| of its pairs at ±3 days (NaN if it has none)."""
    mean_abs = np.abs(st['coh'][pol][st['good']]).mean(axis=0)
    nb = mean_abs.shape[0]
    return np.array([np.mean([mean_abs[i, j] for j in range(nb) if abs(lag_days(st, i, j)) == 3] or [np.nan])
                     for i in range(nb)])


def filter_acquisitions(st):
    """st['kept'][pol] = acquisitions whose 3-day |γ| reaches mean - N_STD * std over the acquisitions; prints the table."""
    m = st['meta']
    pols = m['coh_pols'] + ['XP']
    st['kept'] = {}
    lines = ['| pol | ' + ' | '.join(f'{k}: {d[5:]}' for k, d in enumerate(m['dates'])) + ' | mean | std | threshold | kept |',
             '|---' * (len(m['dates']) + 5) + '|']
    for pol in pols:
        c3 = three_day_coherence(st, pol)
        threshold = np.nanmean(c3) - N_STD * np.nanstd(c3)
        kept = [k for k, v in enumerate(c3) if v >= threshold]
        st['kept'][pol] = kept
        cells = [f'{v:.3f}' if k in kept else f'~~{v:.3f}~~' for k, v in enumerate(c3)]
        lines.append(f'| {pol} | ' + ' | '.join(cells) + f' | {np.nanmean(c3):.3f} | {np.nanstd(c3):.3f} | '
                     f'**{threshold:.3f}** | {len(kept)}/{len(c3)} |')
    display(Markdown(f'**Acquisition filter**: mean 3-day |γ| per acquisition, struck through = below '
                     f'mean - {N_STD:g} std (dropped)\n\n' + '\n'.join(lines)))


def stack_summary(st):
    m, w = st['meta'], st['meta']['window']
    rows = '\n'.join(f"| {k} | {d} | {kz:+.3f} |{' primary' if k == m['primary_index'] else ''}"
                     for k, (d, kz) in enumerate(zip(m['dates'], m['kz_mean_rad_per_m'])))
    return (f"## {st['name']}\n\n"
            f"{m['orbit_state']}, {len(m['dates'])} acquisitions. Window {w['NW_a']} x {w['NW_r']} px = "
            f"**{w['window_ground_az_m']:.0f} x {w['window_ground_rg_m']:.0f} m** (az x rg, rg/az = {w['aspect_rg_over_az']:.2f}), "
            f"{w['looks']} looks, output grid {w['output_spacing_az_m']:.0f} x {w['output_spacing_rg_m']:.0f} m, "
            f"{int(st['good'].sum())} valid cells.\n\n"
            f"| idx | date | mean kz [rad/m] |\n|---|---|---|\n{rows}")


def plot_mean_matrix(st):
    """Mean over the valid cells of the 3Nb x 3Nb MBMP correlation: |γ| and phase of the mean γ."""
    m = st['meta']
    nb, pols = len(m['dates']), m['mbmp_pols']
    corr = st['mbmp_corr'][st['good']]                 # (Ncells, 3Nb, 3Nb)
    mean_abs = np.abs(corr).mean(axis=0)
    mean_phase = np.rad2deg(np.angle(corr.mean(axis=0)))
    labels = [f'{p} {d[5:]}' for p in pols for d in m['dates']]
    n = len(labels)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10), layout='constrained')
    panels = [(mean_abs, 'Blues', (0, 1), 'mean |γ|'),
              (mean_phase, 'twilight', (-180, 180), 'phase of mean γ [deg]')]
    for ax, (mat, cmap, (vmin, vmax), title) in zip(axes, panels):
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax)
        for b in range(1, len(pols)):                  # polarisation block borders
            ax.axhline(b * nb - 0.5, color='black', lw=1.5)
            ax.axvline(b * nb - 0.5, color='black', lw=1.5)
        ax.set_xticks(range(n), labels, rotation=90, fontsize=8)
        ax.set_yticks(range(n), labels, fontsize=8)
        ax.set_title(title, fontsize=13)
        fig.colorbar(im, ax=ax, shrink=0.8)
    for i in range(n):                                 # values of mean |γ|
        for j in range(n):
            v = mean_abs[i, j]
            axes[0].text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                         color='white' if v > 0.55 else 'black')
    fig.suptitle(f"{st['name']} — MBMP correlation {n} x {n} (HH, XP, VV x {nb} acquisitions), "
                 f"mean over {corr.shape[0]} cells", fontsize=14)
    plt.show()


def plot_pair_grid(st, pol):
    """Grid of the kept acquisitions of one polarisation: |γ| maps above the diagonal, intensity on it."""
    m = st['meta']
    kept = st['kept'][pol]
    if len(kept) < 2:
        print(f"{st['name']} {pol}: {len(kept)} acquisition(s) above the 3-day threshold, no pairs to show")
        return
    p = m['coh_pols'].index(pol)
    coh = st['coh'][pol]
    power_db = 10 * np.log10(st['power_pol'][:, :, p] + 1e-12)
    db_lim = np.nanpercentile(power_db[st['good']], [2, 98])
    n = len(kept)

    fig, axes = plt.subplots(n, n, figsize=(2.6 * n + 1.5, 2.6 * n * min(st['aspect'], 1.6) + 1),
                             layout='constrained', squeeze=False)
    for a, i in enumerate(kept):
        for b, j in enumerate(kept):
            ax = axes[a, b]
            if a > b:
                ax.axis('off')
                continue
            if a == b:
                img, kw = power_db[:, :, i], dict(cmap='gray', vmin=db_lim[0], vmax=db_lim[1])
                txt = f'{m["dates"][i]}  [dB]'
            else:
                img, kw = np.abs(coh[:, :, i, j]), dict(cmap='Blues', vmin=0, vmax=1)
                txt = f'|γ| {i}-{j} ({lag_days(st, i, j)} d)  μ={np.nanmean(img[st["good"]]):.2f}'
            im = show_map(ax, st, img, labels=(a == b), **kw)
            ax.set_title(txt, fontsize=8)
            if a == b == 0: im_db = im
            if (a, b) == (0, 1): im_abs = im
    # colorbars in the empty lower triangle
    half = max(n // 2, 1)
    fig.colorbar(im_abs, ax=axes[-1, :half], location='bottom', shrink=0.9, label='|γ|')
    fig.colorbar(im_db, ax=axes[-1, half:] if n > 1 else axes[-1, :], location='bottom', shrink=0.9,
                 label='intensity [dB]')
    fig.suptitle(f"{st['name']} — {pol}: |γ| (upper), intensity (diagonal), {n}/{len(m['dates'])} acquisitions kept  "
                 f"[crop pixels: rows = azimuth, columns = slant range]", fontsize=13)
    plt.show()


def rgb_anchors(st, pol):
    """Kept acquisitions i whose partners at +3, +6, +9 days are kept too -> [(i, [j3, j6, j9]), ...]"""
    kept = st['kept'][pol]
    out = []
    for i in kept:
        partners = [next((j for j in kept if lag_days(st, i, j) == lag), None) for lag in RGB_LAGS_DAYS]
        if None not in partners:
            out.append((i, partners))
    return out


def plot_rgb(st):
    """One RGB image per valid anchor and per pol: R, G, B = |γ| at 3, 6, 9 days."""
    m = st['meta']
    anchors = {pol: rgb_anchors(st, pol) for pol in RGB_POLS}
    n_col = max(len(a) for a in anchors.values())
    if n_col == 0:
        print(f"{st['name']}: no kept acquisition has kept partners at {RGB_LAGS_DAYS} days, no RGB to show")
        return
    lag_txt = '/'.join(map(str, RGB_LAGS_DAYS))
    fig, axes = plt.subplots(len(RGB_POLS), n_col, squeeze=False, layout='constrained',
                             figsize=(3.2 * n_col + 1, 3.2 * len(RGB_POLS) * min(st['aspect'], 1.6) + 0.6))
    for r, pol in enumerate(RGB_POLS):
        for c in range(n_col):
            ax = axes[r, c]
            if c >= len(anchors[pol]):
                ax.axis('off')
                continue
            i, partners = anchors[pol][c]
            # absolute 0-1 scale: each channel is the raw |γ| (no stretch to the max of the image),
            # imshow shows float RGB as-is, so a channel at 0.3 is 30 % intensity in every image
            rgb = np.stack([np.abs(st['coh'][pol][:, :, i, j]) for j in partners], axis=-1)
            rgba = np.concatenate([np.clip(rgb, 0, 1), st['good'][..., None].astype(float)], axis=-1)
            show_map(ax, st, rgba, labels=(c == 0))
            means = rgb[st['good']].mean(axis=0)
            ax.set_title(f"{pol}  {m['dates'][i]} ({i} -> {', '.join(map(str, partners))})\n"
                         f"mean R {means[0]:.2f}  G {means[1]:.2f}  B {means[2]:.2f}", fontsize=8)
        if len(anchors[pol]) == 0:
            axes[r, 0].text(0.5, 0.5, f'{pol}: no valid anchor', ha='center', va='center',
                            transform=axes[r, 0].transAxes)
    fig.suptitle(f"{st['name']} — RGB of |γ|: R = {RGB_LAGS_DAYS[0]} d, G = {RGB_LAGS_DAYS[1]} d, "
                 f"B = {RGB_LAGS_DAYS[2]} d  (white = stable over {lag_txt} d, red = fast decorrelation)",
                 fontsize=12)
    plt.show()

## Per-stack results

Each stack runs the same sequence: summary, mean 21 x 21 MBMP matrix (all acquisitions), acquisition filter,
pair grids of the kept acquisitions for HH, HV, VH, VV, RGB of the 3 / 6 / 9 day coherence for HH, XP, VV.

In [ ]:
for path in files:
    st = load_stack(path)
    display(Markdown(stack_summary(st)))
    plot_mean_matrix(st)
    filter_acquisitions(st)
    for pol in ['HH', 'HV', 'VH', 'VV']:
        plot_pair_grid(st, pol)
    plot_rgb(st)
    del st

## |γ| vs temporal baseline (HH)

When the cell runs it asks for a ROI box in crop pixels (`az_min az_max rg_min rg_max`, read the pixel numbers off the
axes of the maps above; blank = whole stack; set `ROI_BOX` to skip the prompt). For each stack the ROI is drawn on the |γ| map, then the
reference acquisition `REF_IDX` paired with each acquisition in `PARTNER_IDX`: |γ| of every valid cell inside the ROI
against the temporal baseline in days. Left: scatter (horizontal jitter, one colour per pair); right: box plot of the same
values (median, quartiles, whiskers at 1.5 IQR, outliers hidden), with the mean marked as a diamond.

In [ ]:
BASELINE_POL = 'HH'
REF_IDX = 2                  # reference acquisition (0-based index in the stack)
PARTNER_IDX = [3, 4, 5]      # partners -> pairs (2, 3), (2, 4), (2, 5)
ROI_BOX = None               # (az_min, az_max, rg_min, rg_max) in crop pixels; None = ask when the cell runs


def ask_roi():
    """ROI_BOX if set, otherwise read 'az_min az_max rg_min rg_max' from the prompt (blank = whole stack)."""
    if ROI_BOX is not None:
        return tuple(ROI_BOX)
    for path in files:
        with np.load(path) as d:
            print(f"{path.parent.name}: azimuth pixels {d['az_index'][0]} .. {d['az_index'][-1]}, "
                  f"range pixels {d['rg_index'][0]} .. {d['rg_index'][-1]}")
    while True:
        txt = input('ROI box "az_min az_max rg_min rg_max" in crop pixels (blank = whole stack): ').replace(',', ' ').split()
        if not txt:
            return None
        try:
            az0, az1, rg0, rg1 = map(int, txt)
            return min(az0, az1), max(az0, az1), min(rg0, rg1), max(rg0, rg1)
        except ValueError:
            print('  need 4 integers, e.g. 3000 5000 300 600')


def roi_mask(st, roi):
    """Valid cells inside the ROI box (all valid cells when roi is None)."""
    if roi is None:
        return st['good']
    az0, az1, rg0, rg1 = roi
    az, rg = st['az_index'][:, None], st['rg_index'][None, :]
    return st['good'] & (az >= az0) & (az <= az1) & (rg >= rg0) & (rg <= rg1)


def plot_roi_overview(st, roi, pol=BASELINE_POL, ref=REF_IDX, partner=PARTNER_IDX[0]):
    """|γ| map of the first pair with the ROI box drawn on it."""
    fig, ax = plt.subplots(figsize=(6.5, 6.5 * min(st['aspect'], 1.6)), layout='constrained')
    im = show_map(ax, st, np.abs(st['coh'][pol][:, :, ref, partner]), cmap='Blues', vmin=0, vmax=1)
    if roi is not None:
        az0, az1, rg0, rg1 = roi
        ax.add_patch(plt.Rectangle((rg0, az0), rg1 - rg0, az1 - az0, fill=False, ec='red', lw=2))
    ax.set_title(f"{st['name']}\n{pol} |γ| {ref}-{partner} with ROI (red)", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8, label='|γ|')
    plt.show()


def plot_coherence_vs_baseline(st, roi=None, pol=BASELINE_POL, ref=REF_IDX, partners=PARTNER_IDX):
    """|γ| of the valid cells inside the ROI for the pairs (ref, j) vs temporal baseline: scatter and box plot."""
    m = st['meta']
    mask = roi_mask(st, roi)
    if not mask.any():
        print(f"{st['name']}: no valid cell inside the ROI {roi}, skipped")
        return
    lags = [abs(lag_days(st, ref, j)) for j in partners]
    values = [np.abs(st['coh'][pol][:, :, ref, j][mask]) for j in partners]
    labels = [f'{ref}-{j}\n{m["dates"][ref][5:]} / {m["dates"][j][5:]}' for j in partners]
    colors = plt.cm.tab10(np.arange(len(partners)))
    rng = np.random.default_rng(0)

    fig, (ax_s, ax_b) = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True, layout='constrained')
    for lag, v, c, lab in zip(lags, values, colors, labels):
        x = lag + rng.uniform(-0.4, 0.4, v.size)
        ax_s.scatter(x, v, s=4, alpha=0.3, color=c, edgecolors='none', label=lab.replace('\n', '  '))
        ax_s.plot(lag, np.median(v), 'k_', ms=30, mew=2)
    ax_s.set_title('scatter (black bar = median)')
    ax_s.legend(markerscale=4, fontsize=8, loc='lower left')

    bp = ax_b.boxplot(values, positions=lags, widths=1.2, showfliers=False, patch_artist=True,
                      showmeans=True, meanprops=dict(marker='D', markerfacecolor='white', markeredgecolor='black'),
                      medianprops=dict(color='black', lw=2))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    for k, (lag, v) in enumerate(zip(lags, values)):
        q1, med, q3 = np.percentile(v, [25, 50, 75])
        top = bp['whiskers'][2 * k + 1].get_ydata().max()      # upper whisker end
        ax_b.annotate(f'med {med:.2f}\nIQR {q3 - q1:.2f}', (lag, top), xytext=(0, 4), textcoords='offset points',
                      ha='center', va='bottom', fontsize=8)
    ax_b.set_title('box plot (whiskers 1.5 IQR, ◇ = mean)')

    for ax in (ax_s, ax_b):
        ax.set_xticks(lags, [f'{l} d\n{lab}' for l, lab in zip(lags, labels)], fontsize=8)
        ax.set_xlim(min(lags) - 2, max(lags) + 2)
        ax.set_xlabel('temporal baseline [days]')
        ax.grid(alpha=0.3)
    ax_s.set_ylabel('|γ|')
    ax_s.set_ylim(0, 1)
    fig.suptitle(f"{st['name']} — {pol} |γ| vs temporal baseline, reference {m['dates'][ref]} (idx {ref}), "
                 f"{values[0].size} cells per pair\n"
                 + ('whole stack' if roi is None else 'ROI azimuth pixels {} .. {}, range pixels {} .. {}'.format(*roi)),
                 fontsize=12)
    plt.show()


roi = ask_roi()
print('ROI:', 'whole stack' if roi is None else roi)
for path in files:
    st = load_stack(path)
    plot_roi_overview(st, roi)
    plot_coherence_vs_baseline(st, roi)
    del st
